In [ ]:

import os
import importlib

import pandas as pd
import numpy as np

import inframind_proteus.outbreak_dynamics
importlib.reload(inframind_proteus.outbreak_dynamics)
from inframind_proteus.outbreak_dynamics import (
    SimulationConfig, RenewalSimulator, SimulationOutput,
    LogisticRT, ConstantGammaGT
)

# os.chdir("..")
# print(os.getcwd())

In [ ]:

simulator = RenewalSimulator(
    rt_model=LogisticRT(),
    gt_model=ConstantGammaGT(
        shape=10.,
        scale=1.8,
    ),
    config=SimulationConfig(

    )
)

simulator

# simulator.config.temporal.sim_start

In [ ]:
config = simulator.config

config.temporal.calibration_start = pd.Timestamp("2023-10-02")
config.temporal.calibration_end =   pd.Timestamp("2023-12-31")

params_df = pd.DataFrame(
    {
        # Generation time
        "gt_gamma_shape":10.0,
        "gt_gamma_scale": 1.8,  # Expectancy = product

        # R(t) stepped logistic function
        "rt_logist_width": 13.,  # In days
        "rt_logist_center": 100.,  # In days
        "rt_logist_roff": 1.0,  # Off-season R value (before start)
        "rt_logist_start": 50.,
        "rt_logist_rmin": 0.2,  # Post-outbreak baseline
        "rt_logist_rmax": 1.6,  # Essentially "R0"
        "rt_logist_end": 150.,

        # Infection-to-notification model
        "notif_nb_overdispersion": 10.,
        "notif_scaling_factor": 1.0,  # Only applied if using external factor
    },
    index=range(config.num_simulations)
)

# # Override with LHS
# from
params_df["rt_logist_rmax"] = np.linspace(0.9, 2.5, config.num_simulations)

initial_infec_df = pd.DataFrame(
    {
        t: np.ones(config.num_simulations) for t in range(0, simulator._gt_max_steps * simulator._step_dt, simulator._step_dt)
    }
)

# --- Observation data
uf = "SP"
tgt_data_df = pd.read_csv(
    f"../.local/dengue/cases_tseries_{uf}.csv",
    parse_dates=["date"]
)
sr = tgt_data_df.set_index("date")["casos"]
date_start = config.temporal.calibration_start
date_end = date_start + pd.Timedelta(weeks=config.num_time_steps + 2)  # Add some buffer to ensure we have enough
sr = sr.loc[date_start:date_end - pd.Timedelta(days=1)]  # Inclusive end, so subtract one day
observations_sr = sr.reset_index(drop=True)

# config.mode = "calibration"
config.mode = "projection"

results = simulator.run(
    params_df=params_df,
    initial_infec_df=initial_infec_df,
    observations_sr=observations_sr,
)
params_df

results

In [ ]:
results.cases_df.agg(["mean", "std"]).T